# Mixed LoRA Hypothesis Check

50% 混合 LoRA について、次を比較する。

- `z_target`: target base LoRA の埋め込み
- `z_donor`: donor base LoRA の埋め込み
- `z_mixed_dataset`: 実際に学習済みの 50% mixed LoRA の埋め込み
- `z_token_mix`: token 列を 0.5:0.5 で線形補間してから encoder に通した埋め込み
- `z_embed_mix`: `0.5 * z_target + 0.5 * z_donor` の埋め込み

mixed 側は calibration をかけてから比較する。

In [14]:
from pathlib import Path
import re
import sys
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

PROJECT_DIR = Path('/root/work-dir/Image_LoRA_encoder')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

ROOT = Path('/root/work-dir')
BASE_COMPRESSED_DIR = ROOT / 'preprocessing_LoRA' / 'compressed_rank32_artLoRA'
MIXED_COMPRESSED_DIR = ROOT / 'preprocessing_LoRA' / 'compressed_rank32_artLoRA_mixed'
MIXED_PLAN_CSV = ROOT / 'WikiArt_LoRA_Dataset_artist100_rank32_mixed' / 'metadata' / 'lora_plan.csv'
CATEGORY_JSON = ROOT / 'WikiArt_LoRA_Dataset_artist100_rank32_mixed' / 'metadata' / 'category_order192_artLoRA.json'
CALIBRATION_PARAMS_DIR = ROOT / 'WikiArt_LoRA_Dataset_artist100_rank32_calibration' / 'calibration' / 'params_rank32_artLoRA'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from config import CONFIG
from dataset.loader import load_model_matrix
from model.aggregator import TokenAggregator
from model.heads import ProjectionHead
from model.lora_encoder import LoraEncoder
from model.transformer_encoder import TransformerEncoder

def latest_saved_model_dir(root: Path) -> Path:
    subdirs = sorted([p for p in root.iterdir() if p.is_dir()])
    if not subdirs:
        raise FileNotFoundError(f'No saved model directories under {root}')
    return subdirs[-1]

def normalize_lora_id(value: str) -> str:
    value = str(value)
    m = re.search(r'(\d+)$', value)
    return m.group(1) if m else value

def load_categories(path: Path) -> list[str]:
    with open(path, 'r', encoding='utf-8') as f:
        categories = json.load(f)
    return [str(category) for category in categories]

def safe_cat_name(category: str) -> str:
    return re.sub(r'[^\w\-\.]', '_', category).rstrip('.')

def calibration_param_path(params_dir: Path, safe_cat: str, dimension: int) -> Path:
    return params_dir / f'calibration_params_{safe_cat}_{dimension}d.npz'

def load_calibration_param_cache(params_dir: Path, category_names: list[str], dimension: int):
    cache = {}
    missing = []
    for category in category_names:
        safe_cat = safe_cat_name(category)
        npz_path = calibration_param_path(params_dir, safe_cat, dimension)
        if not npz_path.exists():
            missing.append(safe_cat)
            continue
        data = np.load(npz_path, allow_pickle=True)
        cache[category] = {
            's': data['s'].astype(np.float32),
            't': data['t'].astype(np.float32),
        }
    if missing:
        print(f'[WARN] missing calibration params for {len(missing)} categories; first 5 = {missing[:5]}')
    return cache

def apply_projection_calibration(proj: np.ndarray, category: str, calibration_params) -> np.ndarray:
    params = calibration_params[category]
    return (proj.astype(np.float32, copy=False) * params['s'] + params['t']).astype(np.float32)

def calibrate_model_matrix_dict(model_mats, category_names, calibration_params):
    out = {}
    for model_id, mat in model_mats.items():
        arr = mat.detach().cpu().numpy().astype(np.float32)
        calibrated_rows = [
            apply_projection_calibration(arr[row_idx], category, calibration_params)
            for row_idx, category in enumerate(category_names)
        ]
        out[model_id] = torch.tensor(np.stack(calibrated_rows), dtype=torch.float32)
    return out

MODEL_DIR = Path('/root/work-dir/Image_LoRA_encoder/saved_models/202605032310')
CATEGORY_NAMES = load_categories(CATEGORY_JSON)
print('DEVICE =', DEVICE)
print('MODEL_DIR =', MODEL_DIR)

DEVICE = cpu
MODEL_DIR = /root/work-dir/Image_LoRA_encoder/saved_models/202605032310


In [15]:
base_mats = load_model_matrix(str(BASE_COMPRESSED_DIR), CONFIG['dimension'])
mixed_mats_raw = load_model_matrix(str(MIXED_COMPRESSED_DIR), CONFIG['dimension'])
calibration_params = load_calibration_param_cache(CALIBRATION_PARAMS_DIR, CATEGORY_NAMES, CONFIG['dimension'])
mixed_mats = calibrate_model_matrix_dict(mixed_mats_raw, CATEGORY_NAMES, calibration_params)

encoder = TransformerEncoder(
    d_model=CONFIG['dimension'],
    num_layers=CONFIG['num_layers'],
    num_heads=CONFIG['num_heads'],
    d_ff=CONFIG['ff_dim'],
    max_len=CONFIG['max_len'],
    use_pos_emb=CONFIG['use_positional_embedding'],
).to(DEVICE)
aggregator = TokenAggregator(
    CONFIG['dimension'],
    hidden_dim=CONFIG['aggregator_hidden_dim'],
    use_mlp=CONFIG['use_mlp_aggregator'],
).to(DEVICE)
lora_enc = LoraEncoder(encoder, aggregator, normalize=True).to(DEVICE)
lora_proj = ProjectionHead(
    in_dim=CONFIG['dimension'],
    out_dim=CONFIG['dimension'],
    head_type=CONFIG.get('lora_proj_type', 'linear'),
    hidden_dim=CONFIG.get('lora_proj_hidden_dim', 256),
    dropout=CONFIG.get('lora_proj_dropout', 0.0),
    normalize=True,
).to(DEVICE)

lora_enc.load_state_dict(torch.load(MODEL_DIR / 'lora_enc.pt', map_location='cpu'))
lora_proj.load_state_dict(torch.load(MODEL_DIR / 'lora_proj.pt', map_location='cpu'))
lora_enc.eval()
lora_proj.eval()

def embed_tokens(token_matrix: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
        x = token_matrix.unsqueeze(0).to(DEVICE)
        z = lora_proj(lora_enc(x))
        return z[0].detach().cpu()

def cosine(a: torch.Tensor, b: torch.Tensor) -> float:
    return float(F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item())

Loaded 160 model matrices
Loaded 12 model matrices


In [24]:
plan_df = pd.read_csv(MIXED_PLAN_CSV)
mixed50_df = plan_df[plan_df['donor_ratio_actual'].round(2) == 0.5].copy()
mixed50_df[['lora_id', 'artist', 'target_base_lora_id', 'donor_base_lora_id', 'donor_ratio_actual']]

,lora_id,artist,target_base_lora_id,donor_base_lora_id,donor_ratio_actual
4,lora_000005,aleksey savrasov,lora_000064,lora_000001,0.5
5,lora_000006,john singer sargent,lora_000008,lora_000001,0.5
6,lora_000007,paul gauguin,lora_000030,lora_000001,0.5
7,lora_000008,vasily vereshchagin,lora_000072,lora_000001,0.5


In [25]:
rows = []

for row in mixed50_df.itertuples(index=False):
    mixed_id_raw = str(row.lora_id)
    target_id_raw = str(row.target_base_lora_id)
    donor_id_raw = str(row.donor_base_lora_id)

    mixed_id = normalize_lora_id(mixed_id_raw)
    target_id = normalize_lora_id(target_id_raw)
    donor_id = normalize_lora_id(donor_id_raw)

    T_target = base_mats[target_id]
    T_donor = base_mats[donor_id]
    T_mixed_dataset = mixed_mats[mixed_id]
    T_token_mix = 0.5 * T_target + 0.5 * T_donor

    z_target = embed_tokens(T_target)
    z_donor = embed_tokens(T_donor)
    z_mixed_dataset = embed_tokens(T_mixed_dataset)
    z_token_mix = embed_tokens(T_token_mix)
    z_embed_mix = F.normalize(0.5 * z_target + 0.5 * z_donor, dim=0)

    rows.append({
        'mixed_id': mixed_id_raw,
        'target_id': target_id_raw,
        'donor_id': donor_id_raw,
        'target_artist': row.target_artist,
        'donor_artist': row.donor_artist,
        'sim(mixed_dataset,target)': cosine(z_mixed_dataset, z_target),
        'sim(mixed_dataset,donor)': cosine(z_mixed_dataset, z_donor),
        'sim(mixed_dataset,token_mix)': cosine(z_mixed_dataset, z_token_mix),
        'sim(mixed_dataset,embed_mix)': cosine(z_mixed_dataset, z_embed_mix),
        'sim(token_mix,embed_mix)': cosine(z_token_mix, z_embed_mix),
        'sim(target,donor)': cosine(z_target, z_donor),
    })

result_df = pd.DataFrame(rows)
result_df

,mixed_id,target_id,donor_id,target_artist,donor_artist,"sim(mixed_dataset,target)","sim(mixed_dataset,donor)","sim(mixed_dataset,token_mix)","sim(mixed_dataset,embed_mix)","sim(token_mix,embed_mix)","sim(target,donor)"
0,lora_000005,lora_000064,lora_000001,aleksey savrasov,vincent van gogh,0.308524,0.492021,0.574582,0.567263,0.991321,-0.004199
1,lora_000006,lora_000008,lora_000001,john singer sargent,vincent van gogh,0.433157,0.216355,0.487328,0.465975,0.988073,-0.028552
2,lora_000007,lora_000030,lora_000001,paul gauguin,vincent van gogh,0.321629,0.405442,0.511919,0.478673,0.986635,0.153578
3,lora_000008,lora_000072,lora_000001,vasily vereshchagin,vincent van gogh,0.255165,0.433216,0.459267,0.450957,0.990083,0.165086


In [26]:
numeric_cols = [c for c in result_df.columns if c.startswith('sim(')]
result_df[numeric_cols].mean().sort_values(ascending=False)

sim(token_mix,embed_mix)        0.989028
sim(mixed_dataset,token_mix)    0.508274
sim(mixed_dataset,embed_mix)    0.490717
sim(mixed_dataset,donor)        0.386759
sim(mixed_dataset,target)       0.329619
sim(target,donor)               0.071478
dtype: float64

In [27]:
all_base_embs = {mid: embed_tokens(tok) for mid, tok in base_mats.items()}

rank_rows = []
for row in mixed50_df.itertuples(index=False):
    mixed_id_raw = str(row.lora_id)
    target_id_raw = str(row.target_base_lora_id)
    donor_id_raw = str(row.donor_base_lora_id)

    mixed_id = normalize_lora_id(mixed_id_raw)
    target_id = normalize_lora_id(target_id_raw)
    donor_id = normalize_lora_id(donor_id_raw)

    z_mixed_dataset = embed_tokens(mixed_mats[mixed_id])
    sims = sorted(
        ((mid, cosine(z_mixed_dataset, emb)) for mid, emb in all_base_embs.items()),
        key=lambda x: x[1],
        reverse=True,
    )
    sorted_ids = [mid for mid, _ in sims]
    rank_rows.append({
        'mixed_id': mixed_id_raw,
        'target_id': target_id_raw,
        'donor_id': donor_id_raw,
        'target_rank_among_base': sorted_ids.index(target_id) + 1,
        'donor_rank_among_base': sorted_ids.index(donor_id) + 1,
        'top5_base_neighbors': [(f'lora_{mid}', score) for mid, score in sims[:5]],
    })

rank_df = pd.DataFrame(rank_rows)
rank_df

,mixed_id,target_id,donor_id,target_rank_among_base,donor_rank_among_base,top5_base_neighbors
0,lora_000005,lora_000064,lora_000001,23,2,"[(lora_000079, 0.5022557377815247), (lora_0000..."
1,lora_000006,lora_000008,lora_000001,2,39,"[(lora_000018, 0.4647357165813446), (lora_0000..."
2,lora_000007,lora_000030,lora_000001,23,6,"[(lora_000004, 0.5187408328056335), (lora_0000..."
3,lora_000008,lora_000072,lora_000001,24,2,"[(lora_000004, 0.5746695399284363), (lora_0000..."


## 読み方

- `sim(mixed_dataset, token_mix)` が高い:
  token 列を 0.5:0.5 で混ぜる仮説が比較的妥当。
- `sim(mixed_dataset, embed_mix)` が高い:
  最終埋め込み空間での線形混合仮説も比較的妥当。
- mixed LoRA の近傍に target / donor の両方が上位で来る:
  mixed retrieval に向けた補助 loss を考える余地がある。
- どれも低い:
  厳密な線形混合ロスより、target / donor への近接を促す緩い loss の方が安全。